# YouTube Video Facial Analysis

This notebook performs demographic analysis of content creators by extracting and analyzing faces from video frames.

**Pipeline**: Video download → Frame extraction → Face detection (InsightFace) → Embedding extraction (MagFace) → Clustering (DBSCAN) → Demographic analysis

**Output**: CSV with age, gender, consistency and quality metrics for each video

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
from tqdm.notebook import tqdm
import subprocess
import time
import scenedetect
import cv2
import torch
import sys
from torch.nn.modules.utils import consume_prefix_in_state_dict_if_present
from insightface.app import FaceAnalysis
from insightface.utils import face_align
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from collections import Counter
from statistics import mode, median

sys.path.append('./videos_files')
from iresnet import iresnet50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
PROJECT_ROOT = os.getcwd()
VIDEO_FOLDER = os.path.join(PROJECT_ROOT, 'videos')
log_videos_path = os.path.join(PROJECT_ROOT, 'videos_log.txt')

# Ensure folders exist
os.makedirs(VIDEO_FOLDER, exist_ok=True)

print('Video folder set to:', VIDEO_FOLDER)

In [ ]:
PROJECT_ROOT = os.path.abspath(os.getcwd())
SUMMARIES_DIR = os.path.join(PROJECT_ROOT, 'summaries')
RESUMMARIES_CSV = os.path.join(SUMMARIES_DIR, 'df_resumos_complete_cleaned.csv')


In [ ]:
WEIGHTS_PATH = os.path.join(PROJECT_ROOT, 'videos_files', 'magface_iresnet50_MS1MV2_ddp_fp32.pth')
caminho_pesos = WEIGHTS_PATH  # kept for backward compatibility
checkpoint = torch.load(caminho_pesos, map_location=device)

model = iresnet50()

if 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

consume_prefix_in_state_dict_if_present(state_dict, prefix="features.module.")

model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

---
## 1. Video Download

Download YouTube videos using **yt-dlp**. A checkpointing mechanism prevents re-downloading videos that have already been processed.

In [ ]:
def save_progress(video_id, output_file):
    with open(output_file, 'a') as f:
        f.write(f"{video_id}\n")
        
def load_progress(output_file):
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            return set(line.strip() for line in f)
    return set()

In [ ]:
completed_videos = load_progress(log_videos_path)

video_folder = os.path.join(project_base_path, 'videos')

In [ ]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Downloading videos", unit="video"):
    output_file = os.path.join(video_folder, f"{video_id}.mp4")

    if os.path.exists(output_file) or video_id in completed_videos:
        print(f"Video {video_id} already downloaded or recorded.")
        continue

    video_url = f"https://www.youtube.com/watch?v={video_id}"

    try:
        print(f"Downloading video {video_id}...")
        command = [
            'yt-dlp',
            '-f', 'bestvideo[vcodec^=avc1][ext=mp4]',
            '-o', output_file,
            '--no-warnings',
            '--retries', '5',
            video_url
        ]

        result = subprocess.run(command, capture_output=True, text=True, check=True)

        save_progress(video_id, log_videos_path)
        print(f"Video {video_id} downloaded successfully.")

    except subprocess.CalledProcessError as e:
        print(f"Error downloading video {video_id}: {e.stderr.strip()}")
        continue

print("All videos processed.")

---
## 2. Frame Extraction and Analysis

This section implements the full facial analysis pipeline:

### Steps:
1. **Face Detection** (`extract_face_frames`): Extracts 1 frame per second and detects faces using InsightFace (buffalo_l), returning aligned faces (112x112), estimated age and gender
2. **Embedding Extraction** (`get_face_features`): Processes each face with the MagFace (iresnet50) model to generate 512-d normalized embeddings, including magnitude as a quality metric
3. **Clustering** (`faces_clustering`): Groups similar faces with DBSCAN (cosine metric) to identify the main presenter vs. secondary appearances
4. **Demographic Analysis** (`get_final_demographics`): Selects the top-5 highest-quality frames from the main cluster and computes:
   - **Age**: Median of detected ages
   - **Gender**: Most frequent gender
   - **Gender Consistency**: Proportion of the dominant gender (1.0 = single gender)
   - **Cluster Percentage**: % of frames the main presenter represents
   - **Quality Score**: Average magnitude of embeddings (detection confidence)

### Output Metrics:
- **age_std**: Standard deviation of ages (age consistency)
- **gender_consistency**: Values < 1 indicate inconsistency
- **cluster_percentage**: High % indicates a single-presenter video


In [ ]:
app = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def extract_face_frames(video_path: str, frames_per_second: int = 1, min_confidence: float = 0.80):
    """
    Extract frames containing detected faces from a video.

    Args:
        video_path: Full path to the video file (.mp4)
        frames_per_second: How many frames to process per second (default: 1)
        min_confidence: Minimum face detection confidence (0-1, default: 0.80)

    Returns:
        List of dicts with:
            - aligned_face: aligned face image (112x112 pixels, NumPy BGR array)
            - age: age estimated by InsightFace (int)
            - gender: estimated gender ('M' or 'F')
    """
    cap = cv2.VideoCapture(video_path)
    fps_original = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps_original == 0:
        print(f"Error: could not get FPS for video {video_path}")
        cap.release()
        return []
    
    # Compute interval between frames to process
    frame_interval = int(fps_original / frames_per_second)
    if frame_interval < 1:
        frame_interval = 1
    
    face_data = []
    frame_count = 0

    while frame_count < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
        success, frame = cap.read()
        
        if not success:
            break

        # Detect faces in the frame
        faces = app.get(frame)
        
        # For each detected face, extract and align
        for face in faces:
            # Filter faces with low confidence
            if face.det_score < min_confidence:
                continue
            
            aligned_face = face_align.norm_crop(frame, landmark=face.kps)
            
            face_data.append({
                'aligned_face': aligned_face,
                'age': face.age,
                'gender': "M" if face.gender == 1 else "F"
            })
            
        frame_count += int(fps_original / frames_per_second)

    cap.release()
    return face_data

In [ ]:
def preprocess_face_for_model(face_img):
    """
    Preprocess a face image for the iresnet50 model.

    Args:
        face_img: face image (NumPy BGR array)

    Returns:
        PyTorch tensor [1, 3, 112, 112] normalized to [-1, 1]
    """
    # Convert BGR to RGB
    face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    
    # Normalize to [-1, 1]
    face_normalized = (face_rgb.astype(np.float32) / 127.5) - 1.0
    
    # Transpose from (H, W, C) to (C, H, W)
    face_transposed = np.transpose(face_normalized, (2, 0, 1))
    
    # Convert to tensor and add batch dimension
    face_tensor = torch.from_numpy(face_transposed).unsqueeze(0).to(device)
    
    return face_tensor


def get_face_features(faces: list):
    """
    Extract feature embeddings from a list of faces.

    Args:
        faces: list of dicts with 'aligned_face', 'age', 'gender'

    Returns:
        List of dicts containing:
            - embedding_norm: normalized embedding (L2)
            - magnitude: magnitude of the raw embedding
            - face_img: original face image
            - age: detected age
            - gender: detected gender
    """
    results = []
    
    for face_data in faces:
        face_img = face_data['aligned_face']
        face_tensor = preprocess_face_for_model(face_img)
        
        with torch.no_grad():
            embedding_tensor = model(face_tensor)
            
            # Compute magnitude of the embedding (useful as a quality metric)
            magnitude = torch.norm(embedding_tensor, p=2).item()
            
            # Convert to numpy
            embedding_raw = embedding_tensor.cpu().numpy().flatten()
            
            # L2 normalization for cosine similarity comparisons
            embedding_norm = embedding_raw / np.linalg.norm(embedding_raw)
            
            results.append({
                'embedding_norm': embedding_norm,
                'magnitude': magnitude,
                'face_img': face_img,
                'age': face_data['age'],
                'gender': face_data['gender']
            })
            
    return results


def faces_clustering(list_embeddings_norm: list, eps: float = 0.4, min_samples: int = 2):
    """
    Cluster facial embeddings using DBSCAN.

    Args:
        list_embeddings_norm: list of normalized embedding arrays
        eps: maximum distance between samples
        min_samples: minimum samples to form a cluster

    Returns:
        Array of labels (-1 for outliers, >=0 for cluster ids)
    """
    if len(list_embeddings_norm) == 0:
        return np.array([])
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine')
    labels = dbscan.fit_predict(list_embeddings_norm)
    
    return labels


def get_final_demographics(features: list, labels: list, top_n: int = 5):
    """
    Use MagFace magnitude to filter top frames and infer age/gender for the main presenter.

    Args:
        features: list with per-frame processing info
        labels: cluster id per frame
        top_n: number of top frames to select
    """
    if len(labels) == 0: return None
    
    # Identify the largest cluster (presenter)
    valid_labels = [l for l in labels if l != -1]
    if not valid_labels: return None
    
    largest_cluster_id = Counter(valid_labels).most_common(1)[0][0]
    
    # Filter faces from this cluster and sort by quality (magnitude)
    cluster_faces = [f for f, l in zip(features, labels) if l == largest_cluster_id]
    cluster_faces.sort(key=lambda x: x['magnitude'], reverse=True)
    
    # Take Top N best frames
    top_n = min(len(cluster_faces), top_n)
    best_samples = cluster_faces[:top_n]
    
    # Compute percentage that largest cluster represents
    total_faces = len(labels)
    largest_cluster_size = len(cluster_faces)
    cluster_percentage = (largest_cluster_size / total_faces) * 100
    
    genders = [s['gender'] for s in best_samples]
    gender_counts = Counter(genders)
    ages = [s['age'] for s in best_samples]
    avg_magnitude = sum(s['magnitude'] for s in best_samples) / top_n
    
    results = {
        'gender': gender_counts.most_common(1)[0][0],
        'age': int(median(ages)),
        'age_std': round(np.std(ages), 2),
        'gender_consistency': round(gender_counts.most_common(1)[0][1] / len(genders), 2),
        'quality_score': round(avg_magnitude, 2),
        'cluster_percentage': round(cluster_percentage, 2)
    }
    
    return results
    
def save_info_csv(video_id: str, csv_path: str, results: dict):
    """
    Save final demographic info to a CSV file.

    Args:
        video_id: unique video identifier
        csv_path: output CSV path
        results: dictionary with results to save
    """    
    gender = results['gender']
    age = results['age']
    age_std = results['age_std']
    gender_consistency = results['gender_consistency']
    quality = results['quality_score']
    cluster_percentage = results['cluster_percentage']
    
    if not os.path.exists(csv_path):
        with open(csv_path, 'w') as f:
            f.write('video_id,age,gender,age_std,gender_consistency,quality_magnitude,cluster_percentage\n')

    with open(csv_path, 'a') as f:
        f.write(f"{video_id},{age},{gender},{age_std},{gender_consistency},{quality},{cluster_percentage}\n")

In [ ]:
csv_path = os.path.join(project_base_path, 'videos_demographics_complete.csv')

# Load already processed videos
processed_videos = set()
if os.path.exists(csv_path):
    df_processed = pd.read_csv(csv_path)
    processed_videos = set(df_processed['video_id'].values)

video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]

for video_filename in tqdm(video_files, desc='Analyzing frames...', unit='video'):
    video_id = os.path.splitext(video_filename)[0]
    
    if video_id in processed_videos:
        continue
    
    video_path = os.path.join(video_folder, video_filename)
    
    try:
        # Extract faces
        faces = extract_face_frames(video_path)
        
        if len(faces) == 0:
            print(f"{video_id}: No faces detected")
            continue
        
        # Extract features
        faces_features = get_face_features(faces)
        
        # Clustering
        embeddings_list = [f['embedding_norm'] for f in faces_features]
        clusters = faces_clustering(embeddings_list)
        
        # Get final demographics
        demographics = get_final_demographics(faces_features, clusters)
        
        if demographics is None:
            print(f"{video_id}: Could not determine demographics")
            continue
        
        # Save to CSV
        save_info_csv(video_id, csv_path, demographics)
        
    except Exception as e:
        print(f"Error processing {video_id}: {str(e)}")
        continue